In [1]:
!unzip STANDUP_RGB_P.zip
!unzip STANDUP_Thermal_P.zip

Archive:  STANDUP_RGB_P.zip
   creating: content/STANDUP_RGB_P/
  inflating: content/STANDUP_RGB_P/H_0098.png  
  inflating: content/STANDUP_RGB_P/R0_0098.png  
  inflating: content/STANDUP_RGB_P/R0_0099.png  
  inflating: content/STANDUP_RGB_P/H_0105.png  
  inflating: content/STANDUP_RGB_P/H_0009.png  
  inflating: content/STANDUP_RGB_P/R0_0012.png  
  inflating: content/STANDUP_RGB_P/R0_0121.png  
  inflating: content/STANDUP_RGB_P/R1_0068.png  
  inflating: content/STANDUP_RGB_P/R1_0037.png  
  inflating: content/STANDUP_RGB_P/R1_0048.png  
  inflating: content/STANDUP_RGB_P/R0_0016.png  
  inflating: content/STANDUP_RGB_P/R1_0026.png  
  inflating: content/STANDUP_RGB_P/R1_0015.png  
  inflating: content/STANDUP_RGB_P/R2_0006.png  
  inflating: content/STANDUP_RGB_P/R0_0107.png  
  inflating: content/STANDUP_RGB_P/R1_0014.png  
  inflating: content/STANDUP_RGB_P/R2_0021.png  
  inflating: content/STANDUP_RGB_P/R2_0015.png  
  inflating: content/STANDUP_RGB_P/H_0078.png  
  inflati

In [2]:
import os, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B3_Weights
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

In [3]:
rgb_meta = pd.read_csv("/content/standup_rgb_metadata.csv")
thermal_feat = pd.read_csv("/content/standup_thermal_features.csv")

ORDINAL_MAP = {"healthy": 0, "R0": 1, "R1": 2, "R2": 3}
NUM_CLASSES = 4

rgb_meta["ordinal_label"] = rgb_meta["label"].map(ORDINAL_MAP)
thermal_feat["ordinal_label"] = thermal_feat["label"].map(ORDINAL_MAP)

print(rgb_meta["ordinal_label"].value_counts().sort_index())

ordinal_label
0    125
1    138
2     68
3     84
Name: count, dtype: int64


In [4]:
patients = rgb_meta[["patient_id", "label"]].drop_duplicates()
train_pid, test_pid = train_test_split(
    patients["patient_id"], test_size=0.2, random_state=SEED,
    stratify=patients["label"]
)

rgb_meta["split"] = rgb_meta["patient_id"].apply(lambda p: "train" if p in set(train_pid) else "test")
thermal_feat["split"] = thermal_feat["patient_id"].apply(lambda p: "train" if p in set(train_pid) else "test")

print(rgb_meta["split"].value_counts())
print(rgb_meta.groupby("split")["patient_id"].nunique())

split
train    331
test      84
Name: count, dtype: int64
split
test      46
train    181
Name: patient_id, dtype: int64


In [5]:
IMG_SIZE, BATCH_SIZE = 224, 32
IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2,0.2,0.1),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class STANDUPDataset(Dataset):
    def __init__(self, df, root, transform):
        self.df, self.root, self.transform = df.reset_index(drop=True), root, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.root, row["filename"])).convert("RGB")
        return self.transform(img), row["ordinal_label"]

train_ds = STANDUPDataset(rgb_meta[rgb_meta.split=="train"], "/content/STANDUP_RGB_P", train_tf)
test_ds  = STANDUPDataset(rgb_meta[rgb_meta.split=="test"], "/content/STANDUP_RGB_P", val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(len(train_ds), len(test_ds))

331 84


In [6]:
class CoralLayer(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_features, 1, bias=False)
        self.bias = nn.Parameter(torch.zeros(num_classes-1))
    def forward(self, x):
        return self.fc(x) + self.bias

def coral_loss(logits, labels, num_classes):
    levels = torch.stack([(labels > k).float() for k in range(num_classes-1)], dim=1).to(logits.device)
    term = F.logsigmoid(logits)*levels + (F.logsigmoid(logits)-logits)*(1-levels)
    return -torch.sum(term, dim=1).mean()

def coral_predict(logits):
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=1)

In [7]:
weights = EfficientNet_B3_Weights.IMAGENET1K_V1
model = models.efficientnet_b3(weights=weights)
for p in model.parameters(): p.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.3)
)
model.coral = CoralLayer(256, NUM_CLASSES)
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:01<00:00, 47.2MB/s]


Trainable params: 393,731


In [8]:
NUM_EPOCHS, LR = 20, 1e-4
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
best_qwk = -1

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        feat = model.classifier(model.features(imgs).mean([2,3]))
        logits = model.coral(feat)
        loss = coral_loss(logits, lbls, NUM_CLASSES)
        loss.backward(); optimizer.step()
    scheduler.step()

    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs = imgs.to(DEVICE)
            feat = model.classifier(model.features(imgs).mean([2,3]))
            logits = model.coral(feat)
            preds.extend(coral_predict(logits).cpu().numpy())
            trues.extend(lbls.numpy())
    acc = accuracy_score(trues, preds); qwk = cohen_kappa_score(trues, preds, weights="quadratic")
    print(f"Epoch {epoch}: Acc={acc:.4f} QWK={qwk:.4f}")
    if qwk > best_qwk:
        best_qwk = qwk
        torch.save(model.state_dict(), "/content/best_rgb_severity_coral.pth")

Epoch 1: Acc=0.3690 QWK=0.2759
Epoch 2: Acc=0.3690 QWK=0.2726
Epoch 3: Acc=0.3929 QWK=0.3273
Epoch 4: Acc=0.4167 QWK=0.3837
Epoch 5: Acc=0.4643 QWK=0.4368
Epoch 6: Acc=0.4524 QWK=0.4899
Epoch 7: Acc=0.4643 QWK=0.5070
Epoch 8: Acc=0.4762 QWK=0.5689
Epoch 9: Acc=0.4643 QWK=0.5394
Epoch 10: Acc=0.4762 QWK=0.5622
Epoch 11: Acc=0.4881 QWK=0.5634
Epoch 12: Acc=0.4762 QWK=0.5493
Epoch 13: Acc=0.4881 QWK=0.5602
Epoch 14: Acc=0.4881 QWK=0.5648
Epoch 15: Acc=0.4881 QWK=0.5554
Epoch 16: Acc=0.4881 QWK=0.5586
Epoch 17: Acc=0.5000 QWK=0.5680
Epoch 18: Acc=0.5000 QWK=0.5646
Epoch 19: Acc=0.4881 QWK=0.5682
Epoch 20: Acc=0.4881 QWK=0.5522


In [9]:
THERMAL_FEATURE_COLS = ["mean_temp","max_temp","std_temp","range_temp","hot_spot_count",
                         "temp_gradient","glcm_contrast","glcm_homogeneity","glcm_entropy"]

train_t = thermal_feat[thermal_feat.split=="train"]
test_t = thermal_feat[thermal_feat.split=="test"]
X_train, y_train = train_t[THERMAL_FEATURE_COLS].values, train_t["ordinal_label"].values
X_test, y_test = test_t[THERMAL_FEATURE_COLS].values, test_t["ordinal_label"].values

binary_classifiers = []
for k in range(NUM_CLASSES-1):
    clf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=SEED, class_weight="balanced")
    clf.fit(X_train, (y_train > k).astype(int))
    binary_classifiers.append(clf)

def ordinal_predict(X, classifiers):
    probs = np.stack([c.predict_proba(X)[:,1] for c in classifiers], axis=1)
    return (probs > 0.5).sum(axis=1)

preds = ordinal_predict(X_test, binary_classifiers)
acc = accuracy_score(y_test, preds); qwk = cohen_kappa_score(y_test, preds, weights="quadratic")
print(f"Thermal Ordinal — Acc={acc:.4f} QWK={qwk:.4f}")

import joblib
joblib.dump(binary_classifiers, "/content/best_thermal_severity_ordinal.joblib")

Thermal Ordinal — Acc=0.3810 QWK=0.4139


['/content/best_thermal_severity_ordinal.joblib']

In [10]:
model.load_state_dict(torch.load("/content/best_rgb_severity_coral.pth"))

for name, param in model.features.named_parameters():
    if "7" in name.split(".")[0] or "8" in name.split(".")[0]:  # last 2 blocks
        param.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.features.parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4},
    {"params": model.coral.parameters(), "lr": 1e-4},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
best_qwk = 0.569  # don't overwrite your existing checkpoint with something worse

for epoch in range(1, 11):
    model.train()
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        feat = model.classifier(model.features(imgs).mean([2,3]))
        logits = model.coral(feat)
        loss = coral_loss(logits, lbls, NUM_CLASSES)
        loss.backward(); optimizer.step()
    scheduler.step()

    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs = imgs.to(DEVICE)
            feat = model.classifier(model.features(imgs).mean([2,3]))
            logits = model.coral(feat)
            preds.extend(coral_predict(logits).cpu().numpy())
            trues.extend(lbls.numpy())
    acc = accuracy_score(trues, preds); qwk = cohen_kappa_score(trues, preds, weights="quadratic")
    print(f"Fine-tune Epoch {epoch}: Acc={acc:.4f} QWK={qwk:.4f}")
    if qwk > best_qwk:
        best_qwk = qwk
        torch.save(model.state_dict(), "/content/best_rgb_severity_coral.pth")

Fine-tune Epoch 1: Acc=0.4643 QWK=0.5367
Fine-tune Epoch 2: Acc=0.5000 QWK=0.5694
Fine-tune Epoch 3: Acc=0.4881 QWK=0.5539
Fine-tune Epoch 4: Acc=0.4881 QWK=0.5492
Fine-tune Epoch 5: Acc=0.4762 QWK=0.5404
Fine-tune Epoch 6: Acc=0.4762 QWK=0.5359
Fine-tune Epoch 7: Acc=0.4762 QWK=0.5280
Fine-tune Epoch 8: Acc=0.4762 QWK=0.5404
Fine-tune Epoch 9: Acc=0.4762 QWK=0.5404
Fine-tune Epoch 10: Acc=0.4762 QWK=0.5404
